# Reforestation Project Status Prediction Model

Training and comparing 5 machine learning models to predict reforestation project outcomes (Successful, Moderate, Failed).

## Part 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

import joblib

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white'})

## Part 2: Load Dataset

In [ ]:
df = pd.read_csv("data/reforestation_projects_1000.csv")
df.head()

## Part 3: Check Dataset

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['Project_Status'].value_counts()

## Part 4: Separate Features

In [ ]:
X = df.drop(columns=[
    "Project_ID",
    "Project_Status"
])

y = df["Project_Status"]

## Part 5: Encode Target

In [ ]:
label = LabelEncoder()

y = label.fit_transform(y)

print("Classes:", label.classes_)
print("Encoded:", np.unique(y))

joblib.dump(label, "models/label_encoder.pkl")

## Part 6: Define Feature Columns

In [ ]:
categorical_features = [
    "Municipality",
    "Soil_Type"
]

numeric_features = [
    'Year',
    'Target_Seedlings',
    'Planted_Seedlings',
    'Survival_Rate',
    'Funding_PHP',
    'Rainfall_mm',
    'Temperature_C',
    'Monitoring_Visits',
    'Pest_Incidents',
    'Fire_Incidents'
]

## Part 7: Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## Part 8: Split Dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Part 9: Train Five Models

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42)
}

trained_models = {}
results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\n{'='*50}")
    print(f"{name} - Accuracy: {acc:.4f}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=label.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
scores = list(results.values())
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
bars = ax.barh(names, scores, color=colors, edgecolor='white', height=0.6)
for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.2%}', va='center', fontweight='bold', fontsize=12)
ax.set_xlabel('Accuracy', fontweight='bold')
ax.set_title('Model Comparison - Reforestation Project Status Prediction', fontweight='bold')
ax.set_xlim(0, 1.0)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Part 11: Select Best Model

In [ ]:
best_name = max(results, key=results.get)
best_score = results[best_name]
best_model = trained_models[best_name]
print(f"\n{'='*50}")
print(f" BEST MODEL: {best_name}")
print(f" Accuracy: {best_score:.4f} ({best_score:.2%})")
print(f"{'='*50}")

## Part 12: Save Artifacts

In [ ]:
joblib.dump(best_model, "models/project_status_model.pkl")
print("✓ Saved project_status_model.pkl")

preprocessor.fit(X_train)
joblib.dump(preprocessor, "models/preprocessor.pkl")
print("✓ Saved preprocessor.pkl")

joblib.dump(X.columns.tolist(), "models/feature_columns.pkl")
print("✓ Saved feature_columns.pkl")

print("\nAll artifacts saved to models/ directory")

## Summary

All 5 models have been trained and evaluated. The best model has been saved as `project_status_model.pkl` along with the preprocessor, label encoder, and feature columns.